**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Adaptive Filtering: the Kalman Filter

The [APA workshop](./Intro_AdFilt_APA.ipynb) adapted a filter by projecting onto recent data. The Kalman filter goes further: it carries a full **statistical belief** about a hidden state — mean *and* uncertainty — and updates that belief optimally with every measurement. It runs in every GPS receiver, drone autopilot, and tracking radar you've ever used.

## 0. Introduction

Setup: a hidden state $\mathbf{x}$ evolves in time; we only see noisy measurements $\mathbf{z}$.

$$\mathbf{x}_k = F\,\mathbf{x}_{k-1} + \mathbf{w}_k \qquad \mathbf{w}_k \sim \mathcal{N}(0, Q)$$
$$\mathbf{z}_k = H\,\mathbf{x}_k + \mathbf{v}_k \qquad\;\; \mathbf{v}_k \sim \mathcal{N}(0, R)$$

$F$: how the state moves. $H$: what we can see of it. $Q$: how much the motion surprises us. $R$: how noisy the sensor is. The filter alternates two beats forever: **predict** (push the belief through $F$, uncertainty grows) and **update** (blend in $\mathbf{z}_k$, uncertainty shrinks).

## 1. Pre-requisites

- [Adaptive Filtering: APA](./Intro_AdFilt_APA.ipynb) — the predict/err/correct loop.
- Basic probability (mean, variance, Gaussian) — see [Random Variables](../Intro_Math/Analysis/README.md#5-random-variables-draft--pending-review) for the rigorous track.
- Matrix multiplication.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(3)

---
### 🕐 Session 1 of 2 — *State-Space Models & the Kalman Equations* (~35 min)
**Goal:** understand the predict/update cycle and why the Kalman gain is the optimal blend.
**Builds on:** [APA workshop](./Intro_AdFilt_APA.ipynb). &nbsp; **Feeds into:** Session 2 (a real tracking problem).

---

## 2. Theory: Predict, Then Update

💡 **Intuition.** You're estimating where a friend is walking. Your **prediction** ("they were heading north at 1 m/s") and a **measurement** ("I glimpsed them over there") disagree. How to combine? Trust each *in proportion to its certainty*. The Kalman gain $K$ is exactly that trust dial, recomputed every step from the two uncertainties: $K \to 1$ when the sensor is sharp and the model vague; $K \to 0$ when the model is sharp and the sensor noisy. Everything else is bookkeeping for "how uncertain am I now?"

### 2.1. The Five Equations

**Predict** (belief moves with the model, uncertainty inflates):
$$\hat{\mathbf{x}}_k^- = F\hat{\mathbf{x}}_{k-1} \qquad P_k^- = F P_{k-1} F^T + Q$$

**Update** (blend in the measurement):
$$K_k = P_k^- H^T (H P_k^- H^T + R)^{-1}$$
$$\hat{\mathbf{x}}_k = \hat{\mathbf{x}}_k^- + K_k(\mathbf{z}_k - H\hat{\mathbf{x}}_k^-) \qquad P_k = (I - K_k H) P_k^-$$

The term $\mathbf{z}_k - H\hat{\mathbf{x}}_k^-$ is the **innovation** — the part of the measurement the model failed to predict. Notice the shape of the state update: *new estimate = prediction + gain × error*. It's the same heartbeat as LMS/NLMS/APA — the Kalman filter is the member of the family whose gain is **derived** rather than tuned.

*(Full derivation — minimizing $\mathrm{tr}(P_k)$ over all linear unbiased estimators — is left for the session discussion; see any of Kay, Haykin, or Simon for the classical proof.)*

### 2.2. Scalar Sanity Check

A constant hidden value measured in noise. The filter should converge to a running average whose uncertainty shrinks like $1/k$.

In [ ]:

# YOUR CODE HERE


---
### 🕐 Session 2 of 2 — *Tracking in Practice* (~40 min)
**Goal:** build a constant-velocity tracker, tune Q and R, and see what mistuning does.
**Builds on:** Session 1.

---

## 3. Application: Tracking a Moving Target

### 3.1. The Model

State = position and velocity, $\mathbf{x} = [p, v]^T$; we measure only position. With time step $\Delta t$:

$$F = \begin{bmatrix} 1 & \Delta t \\ 0 & 1 \end{bmatrix}, \quad H = [1 \;\; 0]$$

"Constant velocity" is a *statistical* statement: velocity persists, plus random acceleration kicks captured by $Q$.

In [ ]:

# YOUR CODE HERE


In [ ]:
        # predict
        # update

# YOUR CODE HERE


The filter also estimates **velocity — a quantity we never measured**. That's the quiet superpower of state-space models: observable combinations of hidden states get estimated for free.

In [ ]:

# YOUR CODE HERE


### 3.2. Tuning $Q$ and $R$

In practice $R$ comes from your sensor's datasheet; $Q$ is the honest confession of how much your model lies. The failure modes:

- **$Q$ too small** — the filter grows overconfident in its model and *ignores* measurements: smooth but lagging, blind to maneuvers.
- **$Q$ too large** — the filter distrusts its model and chases every noisy measurement: jittery, barely better than raw data.

In [ ]:

# YOUR CODE HERE


## 4. Conclusion

The Kalman filter is the predict/correct loop of the [APA workshop](./Intro_AdFilt_APA.ipynb) elevated to a *belief update*: gains derived from modeled uncertainty instead of tuned by hand — optimal when the model is linear and the noise Gaussian. When those assumptions crack, the ideas generalize (EKF/UKF for nonlinearity, and particle filters beyond that).

---
## Where next

- [Recurrent Neural Networks](./README.md#workshop-3--recurrent-neural-networks-available) — an RNN is a *learned, nonlinear* state-space model; the hidden state lives on.
- [Measure Theory → Random Variables](../Intro_Math/Analysis/README.md) — the probability foundations under $Q$, $R$, and "optimal."